# Classificação de Imagens com Redes Neurais Convolucionais (CNN)

**Disciplina:** Aprendizado Profundo  
**Tema:** Redes Neurais Convolucionais — reconhecimento de imagens

---

Implementação de CNNs para classificação de imagens no dataset **CIFAR-10**, com três arquiteturas progressivamente mais complexas:

| Modelo | Arquitetura |
|--------|-------------|
| **Base** | 1 bloco Conv2D + MaxPooling |
| **Aprofundado** | 3 blocos Conv2D + MaxPooling |
| **Com Dropout** | 3 blocos Conv2D + MaxPooling + Dropout 20% |

### Dataset — CIFAR-10
- 60.000 imagens coloridas RGB de 32×32 pixels
- 10 classes: `avião`, `carro`, `pássaro`, `gato`, `cervo`, `cachorro`, `sapo`, `cavalo`, `navio`, `caminhão`
- Divisão: 50.000 treino / 10.000 teste

---
## Imports

In [ ]:
import numpy as np
import tensorflow as tf

from matplotlib import pyplot as plt
from sklearn.model_selection import KFold
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout
from tensorflow.keras.optimizers import Adam
from PIL import Image
import requests
from io import BytesIO

---
## Carregamento e Pré-processamento

In [ ]:
# Carregando o CIFAR-10 diretamente do TensorFlow
(trainX, trainY), (testX, testY) = cifar10.load_data()

# Nomes das 10 classes na ordem dos índices (0–9)
class_names = ['avião','carro','pássaro','gato','cervo',
               'cachorro','sapo','cavalo','navio','caminhão']

In [ ]:
# Visualizando amostras do dataset
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(trainX[i])
    ax.set_title(class_names[trainY[i][0]])
    ax.axis('off')
plt.suptitle('Amostras do CIFAR-10')
plt.tight_layout()
plt.show()

In [ ]:
# Normalização: converte pixels de [0–255] para [0–1]
# Valores menores estabilizam e aceleram o treinamento
trainX = trainX.astype('float32') / 255.0
testX  = testX.astype('float32')  / 255.0

# One-hot encoding: converte rótulo numérico em vetor binário
# Ex: classe 3 → [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
# Necessário para a função de perda categorical_crossentropy
trainY = to_categorical(trainY, 10)
testY  = to_categorical(testY,  10)

print('Treino:', trainX.shape, trainY.shape)
print('Teste: ', testX.shape,  testY.shape)

---
## Treinamento com K-Fold Cross-Validation

O K-Fold divide os dados em `k` partes. Em cada iteração, `k-1` partes treinam e 1 valida. O processo se repete `k` vezes, retornando média e desvio padrão da acurácia.

In [ ]:
# EarlyStopping: interrompe o treino se val_loss não melhorar por 3 épocas consecutivas
# restore_best_weights: restaura os pesos da melhor época ao parar
callback = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=3, restore_best_weights=True
)

def run_kfold(define_model_fn, trainX, trainY, k=5, epochs=10, batch_size=32):
    scores = []
    histories = []

    # shuffle=True garante que os folds não sigam a ordem original dos dados
    kfold = KFold(k, shuffle=True, random_state=1)

    for fold, (train_ix, val_ix) in enumerate(kfold.split(trainX)):
        print(f'\n--- Fold {fold+1}/{k} ---')

        # Novo modelo a cada fold para evitar vazamento de informação entre folds
        model = define_model_fn()

        train_data, train_target = trainX[train_ix], trainY[train_ix]
        val_data,   val_target   = trainX[val_ix],   trainY[val_ix]

        # batch_size=32: número de amostras processadas antes de atualizar os pesos
        history = model.fit(train_data, train_target,
                            epochs=epochs, batch_size=batch_size,
                            validation_data=(val_data, val_target),
                            verbose=1, callbacks=[callback])

        _, acc = model.evaluate(val_data, val_target, verbose=0)
        print(f'Acurácia fold {fold+1}: {acc*100:.3f}%')
        scores.append(acc)
        histories.append(history)

    print(f'\nAcurácia: média={np.mean(scores)*100:.3f}%  desvio={np.std(scores)*100:.3f}%')
    return scores, histories, model

---
## Modelo Base

In [ ]:
def define_model_base():
    model = Sequential([
        # Conv2D: filtros 3x3 deslizam sobre a imagem detectando padrões locais
        # relu: zera valores negativos, introduzindo não-linearidade na rede
        # padding='same': mantém as dimensões espaciais após a convolução
        Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),

        # MaxPooling2D: reduz o mapa de 32x32 para 16x16
        # Seleciona o valor máximo em cada janela 2x2 — preserva as features mais fortes
        MaxPooling2D((2, 2)),

        # Flatten: transforma o mapa 2D em vetor 1D para conectar às camadas densas
        Flatten(),

        # Dense: camada totalmente conectada — combina as features extraídas
        Dense(100, activation='relu'),

        # Softmax: garante que as 10 saídas sejam probabilidades que somam 1
        Dense(10, activation='softmax')
    ])

    # Adam: otimizador adaptativo com ajuste automático de taxa de aprendizado
    # categorical_crossentropy: função de perda para classificação multiclasse com one-hot
    model.compile(optimizer=Adam(learning_rate=0.001),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

define_model_base().summary()

In [ ]:
scores_base, histories_base, model_base = run_kfold(define_model_base, trainX, trainY)

---
## Modelo Aprofundado — 3 blocos Conv2D + MaxPooling

In [ ]:
def define_model_deep():
    model = Sequential([
        # Bloco 1: detecta features simples — bordas e gradientes de cor
        Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
        MaxPooling2D((2, 2)),   # 32x32 → 16x16

        # Bloco 2: combina as features do bloco anterior em texturas e formas
        # 64 filtros: maior capacidade para representar combinações mais complexas
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        MaxPooling2D((2, 2)),   # 16x16 → 8x8

        # Bloco 3: aprende representações de alto nível — partes de objetos
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        MaxPooling2D((2, 2)),   # 8x8 → 4x4

        # Flatten: (4,4,64) → vetor de 1024 valores
        Flatten(),

        # Dense com 128 neurônios para comportar a maior riqueza de features
        Dense(128, activation='relu'),
        Dense(10, activation='softmax')
    ])

    model.compile(optimizer=Adam(learning_rate=0.001),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

define_model_deep().summary()

In [ ]:
scores_deep, histories_deep, model_deep = run_kfold(define_model_deep, trainX, trainY)

---
## Modelo com Dropout — 20% após cada MaxPooling

In [ ]:
def define_model_dropout():
    model = Sequential([
        # Bloco 1
        Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
        MaxPooling2D((2, 2)),
        # Dropout(0.2): desativa 20% dos neurônios aleatoriamente a cada batch de treino
        # Impede que a rede memorize os dados — força aprendizado de padrões gerais
        Dropout(0.2),

        # Bloco 2
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        MaxPooling2D((2, 2)),
        Dropout(0.2),

        # Bloco 3
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        MaxPooling2D((2, 2)),
        # Dropout antes do Flatten reduz overfitting na transição para as camadas densas
        Dropout(0.2),

        Flatten(),
        Dense(128, activation='relu'),
        Dense(10, activation='softmax')
    ])

    model.compile(optimizer=Adam(learning_rate=0.001),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

define_model_dropout().summary()

In [ ]:
scores_dropout, histories_dropout, model_dropout = run_kfold(define_model_dropout, trainX, trainY)

---
## Avaliação no Conjunto de Teste

In [ ]:
# Avaliação final no conjunto de teste — dados nunca vistos durante o treino
_, test_acc = model_dropout.evaluate(testX, testY, verbose=0)
print(f'Acurácia no conjunto de teste: {test_acc*100:.2f}%')

---
## Testando com uma imagem do dataset

In [ ]:
idx = 200

plt.imshow(testX[idx])
plt.axis('off')
plt.title(f'Imagem #{idx}')
plt.show()

# expand_dims adiciona dimensão de batch: (32,32,3) → (1,32,32,3)
# A CNN espera uma lista de imagens, mesmo que seja só uma
pred = model_dropout.predict(np.expand_dims(testX[idx], axis=0))
classe_predita = np.argmax(pred)

print(f'Classe predita: {class_names[classe_predita]}')
print(f'Classe real:    {class_names[np.argmax(testY[idx])]}')
print(f'\nProbabilidades:')
for nome, prob in zip(class_names, pred[0]):
    print(f'  {nome:10s}: {prob*100:.1f}%')

---
## Testando com uma imagem própria

Você pode usar um arquivo local ou uma URL pública.

> **Atenção:** o modelo reconhece apenas as 10 classes do CIFAR-10.  
> A imagem será redimensionada para 32×32 pixels automaticamente.

In [ ]:
# Caminho da imagem — arquivo local ou URL pública
# Para usar URL: url_ou_caminho = 'https://...'
url_ou_caminho = 'passaro.jpg'

# Carrega a imagem
if url_ou_caminho.startswith('http'):
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url_ou_caminho, headers=headers)
    img = Image.open(BytesIO(response.content)).convert('RGB')
else:
    img = Image.open(url_ou_caminho).convert('RGB')

# Redimensiona para 32x32 — tamanho esperado pela CNN
img_resized = img.resize((32, 32))

plt.imshow(img)
plt.axis('off')
plt.title('Imagem de teste')
plt.show()

# Converte para array numpy e normaliza para [0–1]
img_array = np.array(img_resized).astype('float32') / 255.0

# Adiciona dimensão de batch: (32,32,3) → (1,32,32,3)
pred = model_dropout.predict(np.expand_dims(img_array, axis=0))
classe_predita = np.argmax(pred)

print(f'Classe predita: {class_names[classe_predita]}')
print(f'\nProbabilidades:')
for nome, prob in zip(class_names, pred[0]):
    bar = '█' * int(prob * 30)
    print(f'  {nome:10s}: {bar} {prob*100:.1f}%')